# End-to-End AI Scoring Test for a Single Category

This notebook provides a simple and flexible execution pipeline to test and debug **a single CSMS category** .

### Instructions:
1. Run the **Setup & Imports** cell.
2. In the **Configuration** cell, set `TARGET_CATEGORY` to the category ID you want to test (e.g., `'8.1'`, `'1.1'`).
3. Run the remaining cells to execute the pipeline and view the formatted evaluation report!

In [ ]:
# 1. Setup & Imports
import os
import sys
import json
import pandas as pd
from datetime import datetime
from IPython.display import display, Markdown
from info_provider import PROVIDER_DEFAULT


sys.path.append(os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')

from engine.pipeline import discover_category_folders, score_category

# Configure sys.stdout for utf-8 if possible
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

USE_PROVIDER = PROVIDER_DEFAULT
    
print("✅ Imports and environment loaded successfully.")

In [ ]:
# ==========================================
# 2. CONFIGURATION
# ==========================================
TARGET_CATEGORY = '8.1'  # <--  Change this to test any category (e.g., '1.1', '2.1', '8.1')
# ==========================================

# Automatically update model_config.json to use the selected provider
config_path = os.path.abspath('../configs/model_config.json')
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    config['active_provider'] = USE_PROVIDER
    
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=4)
    
    active_model = config.get('providers', {}).get(USE_PROVIDER, {}).get('model', 'unknown')
    print(f" LLM Provider Model Activated")
else:
    print("⚠️ model_config.json not found! Using default provider from environment.")

In [ ]:
# 3. Discover Category Folder
categories = discover_category_folders()
target_info = next((cat for cat in categories if cat['category_id'] == TARGET_CATEGORY), None)

if target_info:
    print(f" Target Category Identified: [{TARGET_CATEGORY}]")
    print(f"    Folder Name: {target_info['folder_name']}")
    # print(f"    Full Path:   {target_info['path']}")
else:
    print(f" Error: Category [{TARGET_CATEGORY}] not found!")
    print("\n Available Categories in Test Submissions:")
    for cat in categories:
        print(f"   - {cat['category_id']}: {cat['folder_name']}")

In [ ]:
# 4. Execute Scoring Pipeline
if target_info:
    print(f"\n Starting AI Scoring Pipeline for Category {TARGET_CATEGORY}...")
    print("=" * 70)
    
    # Run the evaluation pipeline
    result = score_category(
        category_id=target_info['category_id'],
        folder_path=target_info['path'],
        silent=False
    )
    
    print("=" * 70)
    print(" Pipeline Execution Completed!")

In [ ]:
# 5. Display Formatted Evaluation Report
if target_info and 'result' in locals():
    # Prepare summary dataframe
    status_raw = result.get("status", "").upper()
    status_icon = "✅ PASSED" if status_raw == "PASSED" else ("⚠️ NEEDS REVIEW" if status_raw == "NEEDS_REVIEW" else "❌ FAILED")
    
    summary_data = [{
        "Category": result.get("category_id"),
        "Folder": result.get("folder_name"),
        "Expected Score": result.get("expected_score"),
        "Predicted Score": result.get("predicted_score"),
        "Confidence": f"{result.get('confidence', 0.0) * 100:.1f}%",
        "Status": status_icon,
        "Evidence Found": len(result.get("evidence_found", [])),
        "Missing Evidence": len(result.get("missing_evidence", [])),
        "Time (s)": result.get("processing_time_sec")
    }]
    
    df_summary = pd.DataFrame(summary_data)
    
    display(Markdown(f"### 📊 Executive Summary: Category {TARGET_CATEGORY}"))
    display(df_summary)
    
    # Display Detailed Breakdown
    display(Markdown("### 🧠 AI Juror Reasoning & Evidence Breakdown"))
    
    print("\n   EVIDENCE FOUND:")
    ev_found = result.get("evidence_found", [])
    if ev_found:
        for ev in ev_found:
            print(f"  ✔ {ev}")
    else:
        print("  (None)")
        
    print("\n   MISSING EVIDENCE:")
    ev_missing = result.get("missing_evidence", [])
    if ev_missing:
        for ev in ev_missing:
            print(f"  ✖ {ev}")
    else:
        print("  (None)")
        
    print("\n   REASONING & CONSENSUS:")
    reasoning = result.get("reasoning", [])
    if reasoning:
        for r in reasoning:
            print(f"  👉 {r}")
    else:
        print("  (None)")
        
    if result.get("errors") or result.get("warnings"):
        print("\n   LOGGED ERRORS / WARNINGS:")
        for err in result.get("errors", []):
            print(f"  [ERROR] {err}")
        for warn in result.get("warnings", []):
            print(f"  [WARN] {warn}")